# Notebook 4: Evaluate Results

**Objective**: Detailed analysis of experiment results

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from experiment.evaluator import AutomaticEvaluator

## 1. Load Results

In [ ]:
results_dir = Path('../data/results')
latest_exp = sorted(results_dir.glob('experiment_*'))[-1]
results_path = latest_exp / 'raw_results.csv'

results_df = pd.read_csv(results_path)
print(f'✅ Loaded {len(results_df)} results from {latest_exp.name}')
results_df.head()

## 2. Aggregate Metrics

In [ ]:
evaluator = AutomaticEvaluator()
agg = evaluator.aggregate_results(results_df.to_dict('records'))

for key, value in agg.items():
    print(f'{key}: {value}')

## 3. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Compliance scores
axes[0,0].hist(results_df['llm_compliance_score'], bins=10)
axes[0,0].set_title('Compliance Score Distribution')

# Helpfulness scores
axes[0,1].hist(results_df['llm_helpfulness_score'], bins=10)
axes[0,1].set_title('Helpfulness Score Distribution')

# Latency overhead
axes[1,0].hist(results_df['latency_overhead_ms'], bins=30)
axes[1,0].set_title('Latency Overhead Distribution')

# Similarity scores
axes[1,1].hist(results_df['similarity_score'], bins=20)
axes[1,1].set_title('Similarity Score Distribution')

plt.tight_layout()
plt.show()

## 4. Compliance vs Helpfulness

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(
    results_df['llm_compliance_score'],
    results_df['llm_helpfulness_score'],
    alpha=0.6,
    c=results_df['contains_restricted_answer'],
    cmap='RdYlGn_r'
)
plt.xlabel('Compliance Score')
plt.ylabel('Helpfulness Score')
plt.title('Trade-off: Compliance vs Helpfulness')
plt.colorbar(label='Contains Leak')
plt.show()

## 5. Error Analysis

In [ ]:
leaks = results_df[results_df['contains_restricted_answer'] == True]
print(f'Leaked cases: {len(leaks)} ({len(leaks)/len(results_df):.1%})')

if len(leaks) > 0:
    print('\nSample leaked cases:')
    for i, row in leaks.head(3).iterrows():
        print(f"\nQuery: {row['query'][:100]}...")
        print(f"Expected: {row['expected_answer']}")
        print(f"Response: {row['avi_response'][:200]}...")

## Summary

✅ Detailed evaluation complete

**Next**: `05_visualize_for_paper.ipynb` to generate publication figures